## Heat Equation Part B 
# <span style="color:blue">Implicit schemes & spectral analysis </span>
---

## Learning objectives

Understand and apply implicit matrix-based methods for time integration of the 1D heat equation.

By the end of this mini-project, students will be able to:
- Formulate the heat equation semi-discretization as a matrix ODE:  
  $$
  \frac{d\mathbf{u}}{dt} = A\mathbf{u}.
  $$
- Implement implicit time-stepping schemes (Backward Euler and Crank–Nicolson for matrix systems.
- Solve the linear systems arising at each time step using standard numerical linear algebra tools.
- Compare explicit vs implicit schemes with respect to:
  - stability,
  - allowable time step size,
  - computational cost,
  - numerical accuracy.
    
- Compute the eigenvalues and eigenvectors of the discrete Laplacian matrix.
- Interpret eigenvectors as spatial heat modes and eigenvalues as decay rates of these modes.
- Explain stability of time integration methods using the spectral radius of the system matrix.
- Recognize diffusion problems as stiff systems and understand why implicit methods are advantageous.
- Recognise that the boundary conditions affects the formation of the matrix and further the eigenmodes.
- Recognise connection to Fourier Series and Discrete Transforms


## Contents
#### 1. Recap from Part A
#### 2. Motivation for implicit methods
#### 3. Implicit time stepping formulas
#### 4. Linear systems interpretation
#### 5. Notes on different methods
#### 6. Implementations of Backward Euler and Crank–Nicolson
#### 7. Spectral interpretation of the discrete heat equation
#### 8. Properties of the discrete Laplacian operator
#### 9. Eigenvalues as decay rates and eigenvectors as heat patterns
#### 10. Connection to Fourier Series and Discrete Transforms
#### 11. Spectral radius and stiffness
#### 12. Why eigenvalues determine time-stepping stability
#### 13. Explicit vs implicit schemes from a spectral viewpoint
#### 14. Note on numerical damping
#### 15. Implementations of eigenvalues, eigenmodes, and spectral interpretation
    

---

## 1. Recap from Part A

We recall that after spatial discretization, the 1D heat equation becomes the matrix ODE:

$$
\frac{d\mathbf{u}}{dt} = A\mathbf{u},
$$

where:

- $A$ is the discrete Laplacian operator,
- Explicit Euler time stepping was used,
- Stability required a severe restriction on the time step size.

---

## 2. Motivation for implicit methods

Explicit Euler methods for diffusion problems suffer from:

- strict time step restrictions,
- stiffness of the resulting ODE system,
- high computational cost for fine spatial grids.

As the spatial step size $h$ becomes smaller, the stability condition forces:

$$
\Delta t \propto h^2 \quad \Rightarrow \quad \Delta t \text{ very small}.
$$

Implicit methods remove this restriction and allow much larger time steps.

---

## 3. Implicit time stepping formulas

Backward Euler and Crank–Nicolson schemes are described in detail  in Wikipedia:  <a href="https://en.wikipedia.org/wiki/Finite_difference_method#Implicit_method" target="_blank">
    <ins>Finite difference methods</ins>.
</a> Here we only state the recursive matrix formulas for successive temperature distributions :


### Backward Euler

$$
(I - \Delta t\, A)\mathbf{u}^{n+1} = \mathbf{u}^{n}.
$$

### Crank–Nicolson

$$
(I - \tfrac{\Delta t}{2} A)\mathbf{u}^{n+1}
=
(I + \tfrac{\Delta t}{2} A)\mathbf{u}^{n}.
$$

---

## 4. Linear systems interpretation
At each time step:
- the unknown is the new temperature vector $ \mathbf{u}^{n+1} $,
- a linear system must be solved.

The system matrix A has important properties:
- symmetric,
- sparse,
- tridiagonal (banded),

which makes it well suited for efficient numerical linear algebra methods.

👉 Implicit methods shift the perspective from time stepping to **solving systems at each step**.


---

## 5. Notes on different methods
Note 1: 
- Both methods are **unconditionally stable** for the heat equation.
- Crank–Nicolson is **2nd order accurate** in time.
- Backward Euler is **1st order but very robust**.
- These implementations intentionally use dense solvers for clarity

*Note 2:* Matrix $A$ is identical to Part A and works for:
- explicit Euler
- backward Euler
- Crank–Nicolson
- eigenvalue analysis

*Note 3*: In this introductory project we use `np.linalg.solve`, which treats matrices as dense.\
For real engineering-scale simulations (large spatial grids, 2D/3D problems), the system matrices are sparse and structured, and much more efficient solvers are available in the SciPy library (e.g. banded and sparse direct solvers, iterative methods).\
These advanced tools are beyond the scope of this project.

---

✅ Stability is no longer restricted by the time step size in the same way as explicit schemes.


## 6. Implementations of Backward Euler and Crank–Nicolson

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Implementations of Backward Euler and Crank–Nicolson

# Initials

L = 1.0                            # rod length
alpha = 1.0                        # thermal diffusivity of the medium (α)
N = 50                             # number of spatial steps 
T = 0.2                            # final time

x = np.linspace(0, L, N+2)         # x-axis discretization (rod coordinates): (start, stop, num)
h = L / (N+1)                      # spatial step size

# Boundary conditions: u(0,t)=u(L,t)=0 (implicit via interior grid only)
# Initial temperature distribution in the rod (sin shape) 
def initial_condition(x):
    return np.sin(np.pi * x)       # sin shape from 0 -> 180 deg,  0<x<1

u0 = initial_condition(x[1:-1])    # initial interior temperature vector (size N)


In [ ]:
# System matrix A
# N×N discrete Laplacian operator (A is same for all three methods)

main_diag = -2 * np.ones(N)        # main diagonal
off_diag  =  1 * np.ones(N-1)      # upper and lower diagonals

A = (alpha / h**2) * (
    np.diag(main_diag) +
    np.diag(off_diag, 1) +
    np.diag(off_diag, -1)
)

# Optional check:
# print(A)


In [ ]:
def solve_backward_euler(u0, A, dt, T):
    """
    Solves du/dt = Au using the Backward Euler implicit method.

    Parameters:
        u0 : initial temperature vector (NumPy array)
        A  : system matrix
        dt : time step
        T  : final time

    Returns:
        history : 2D NumPy array (rows = time steps, columns = spatial grid points)
    """
    u = u0.copy()
    history = [u.copy()]

    N = len(u0)
    I = np.eye(N)                         # 2-D array with ones on the diagonal and zeros elsewhere

    # Matrix for the linear system: (I - dt*A)
    M = I - dt * A

    steps = int(T / dt)

    for _ in range(steps):
        # Solve: (I - dt*A)u^{n+1} = u^n
        u = np.linalg.solve(M, u)
        history.append(u.copy())

    return np.array(history)


In [ ]:
def solve_crank_nicolson(u0, A, dt, T):
    """
    Solves du/dt = Au using the Crank–Nicolson implicit method.

    Parameters:
        u0 : initial temperature vector (NumPy array)
        A  : system matrix
        dt : time step
        T  : final time

    Returns:
        history : 2D NumPy array (rows = time steps, columns = spatial grid points)
    """
    u = u0.copy()
    history = [u.copy()]

    N = len(u0)
    I = np.eye(N)

    # Left-hand and right-hand matrices
    M_left  = I - 0.5 * dt * A
    M_right = I + 0.5 * dt * A

    steps = int(T / dt)

    for _ in range(steps):
        rhs = M_right @ u
        u = np.linalg.solve(M_left, rhs)
        history.append(u.copy())

    return np.array(history)


In [ ]:
# Call solve functions

dt = 5 * h**2 / alpha                         # time step, much larger than explicit stability limit

U_be = solve_backward_euler(u0, A, dt, T)
U_cn = solve_crank_nicolson(u0, A, dt, T)


In [ ]:
# Explicit Euler solution from PartA (for comparison)

def solve_explicit(u0, A, dt, T):           
    u = u0.copy()                                                               
    U_history = [u.copy()]                  
    steps = int(np.round(T/dt))             
    for _ in range(steps):                  
        u = u + dt*A@u                      
        U_history.append(u.copy())          
    
    return np.array(U_history)             
  
dt = 0.4 * h**2 / alpha                    # time step: stabil for explicit Euler
#dt = 5 * h**2 / alpha                     # TRY THIS: time step, much larger than explicit stability limit 
u_exp = solve_explicit(u0, A, dt, T)       # uses the same large dt (expected to be unstable)

In [ ]:
# Plot final temperature profiles

fig, ax = plt.subplots(figsize=(7, 4))

plt.plot(x[1:-1], u_exp[-1], color="lightblue", linewidth=3.0, label="Explicit Euler (unstable)")
plt.plot(x[1:-1], U_be[-1], color="g", linewidth=0.7, label="Backward Euler (stable)")
plt.plot(x[1:-1], U_cn[-1], color="r", linewidth=0.7,  label="Crank–Nicolson (stable)")

ax.set_xlabel("x")
ax.set_ylabel("temperature")
ax.set_title("Final temperature profiles (large time step)")
ax.legend();
ax.grid()

plt.show()        # Note the temperature scale 0.0 - 0.14 (initial max temp in rod 1.0 and scale 0.0 -1.0; see next plot)


**Interpretation.**
Note the scaling of the temperature scale; the height of the original peak is 1.0.
The rod has not yet cooled completely. The original sinusoidal shape of the temperature distribution has been preserved.

👉 The matrix formulation encodes the full system behavior at each time level.


In [ ]:
# Plot time evolution snapshots of Backward Euler solution

fig, ax = plt.subplots(figsize=(7, 4))

snapshots = [0, len(U_be)//4, len(U_be)//2, -1]

for k in snapshots:
    plt.plot(x[1:-1], U_be[k], label=f"t-step {k}")

ax.set_xlabel("x")
ax.set_ylabel("temperature")
ax.set_title("Backward Euler solution over time")
ax.legend(); ax.grid()

plt.show()


In [ ]:
# Norm comparison over time (stability visualization)
# This shows numerical blow-up clearly.

norm_exp = np.linalg.norm(u_exp, axis=1)
norm_be   = np.linalg.norm(U_be, axis=1)
norm_cn   = np.linalg.norm(U_cn, axis=1)

fig, ax = plt.subplots(figsize=(7, 4))

plt.plot(norm_exp, label="Explicit Euler")
plt.plot(norm_be,   label="Backward Euler",linewidth=3.0)
plt.plot(norm_cn,   label="Crank–Nicolson")

ax.set_xlabel("time step")
ax.set_ylabel("||u||₂")
ax.set_title("Solution norm vs time")
ax.set_yscale("log")
ax.legend()
ax.grid()

plt.show()


### Interpretation

For the large time step used here:

- Explicit Euler becomes unstable and blows up.
- Backward Euler remains stable for arbitrarily large time steps, but introduces excess numerical damping of high-frequency spatial modes, leading to overly smooth solutions for large Δt.
- Crank–Nicolson remains stable and is more accurate (less diffusive). (In this plot the B_E and C-N lines overlap)

This illustrates:

- the CFL stability restriction of explicit methods,
- the unconditional stability of implicit schemes,
- the trade-off between stability and accuracy.



---

## 7. Spectral interpretation of the discrete heat equation


After spatial discretization, the 1D heat equation takes the matrix form:

$$
\frac{d\mathbf{u}}{dt} = A \mathbf{u},
$$

where:

- $\mathbf{u}(t)$ is the vector of temperatures at the spatial grid points,
- $A$ is the discrete Laplacian operator.

The behavior of this system is completely determined by the **eigenvalues and eigenvectors of the matrix $A$**.

This provides a powerful link between:

- heat diffusion (physics),
- time integration methods (numerical analysis),
- and spectral properties of matrices (linear algebra).


✅ Spectral analysis provides a powerful tool to understand stability and dynamics.


---

### 8. Properties of the discrete Laplacian operator

The matrix $A$ obtained from the second-derivative finite difference approximation has the following properties:

- tridiagonal,
- symmetric,
- negative definite.


**Consequences**:
- all eigenvalues of $A$ are real and negative,
- its eigenvectors are orthogonal,
- the eigenvectors form an orthogonal basis of $\mathbb{R}^N$.

**Physically, this means**:

✅  Every spatial temperature distribution can be **decomposed into independent modes**, and **each mode decays in time independently**.


---

## 9. Eigenvalues as decay rates and eigenvectors as heat patterns

The eigenvalue problem for the system matrix is:

$$
A \mathbf{v}_k = \lambda_k \mathbf{v}_k .
$$

Here:

- $\mathbf{v}_k$ is the $k$-th **eigenvector** (a spatial temperature pattern),
- $\lambda_k < 0$ is the corresponding **eigenvalue**.

If the solution is decomposed as

$$
\mathbf{u}(t) = \sum_k c_k(t)\,\mathbf{v}_k,
$$

then each coefficient evolves independently as:

$$
c_k(t) = c_k(0)\, e^{\lambda_k t}.
$$

Therefore:

- eigenvectors represent spatial heat modes,
- eigenvalues represent **decay rates** of these modes.

Low-frequency modes decay slowly.  
High-frequency (oscillatory) modes decay rapidly.


👉 **Eigenvalues** determine **how different modes evolve** and whether they grow or decay.


---

## 10. Connection to Fourier Series and Discrete Transforms

A useful parallel exists between continuous PDEs and their discrete counterparts:

**Continuous setting:**  
The eigenfunctions of differential operators such as the Laplacian are sine and cosine waves. These form the foundation of Fourier series.

**Discrete setting:**  
The eigenvectors of the discrete Laplacian matrix are sampled versions of those same sine (or cosine) waves, depending on the imposed boundary conditions.

These discrete eigenvectors are closely related to the bases used in discrete sine and cosine transforms, which are themselves closely connected to the Discrete Fourier Transform (DFT).

This correspondence explains why Fourier-based ideas naturally arise in both continuous PDE analysis and numerical discretizations, and why spectral methods are so effective for diffusion-type problems.


---

## 11. Spectral radius and stiffness

The **spectral radius** of a matrix is defined as:

$$
\rho(A) = \max_k |\lambda_k|.
$$

For the discrete Laplacian:

- $\rho(A)$ increases as the spatial grid is refined,
- approximately $\rho(A) \sim \mathcal{O}(1/h^2)$.

Therefore refining the spatial grid makes the system more stiff.

Implications:

- large $\rho(A)$ → fast decaying modes,
- fast modes + slow modes together → **stiff system**.

This stiffness explains why explicit time stepping methods require very small time steps for stability.


---

## 12. Why eigenvalues determine time-stepping stability

For any time integration method, each eigenmode evolves independently.

If the numerical scheme updates one time step as

$$
u^{n+1} = G u^n
$$

where $G $ is the iteration matrix,

then each eigenmode is multiplied by a factor:

$$
g_k = g(\lambda_k \Delta t),
$$

called the **amplification factor**.

Stability requires:

$$
|g_k| \le 1 \quad \text{for all eigenvalues } \lambda_k.
$$

Thus, stability depends directly on the spectrum of the system matrix $A$.


---

## 13. Explicit vs implicit schemes from a spectral viewpoint

For the heat equation:

**Explicit Euler:**

$$
g_k = 1 + \Delta t\,\lambda_k
$$

Stability requires:

$$
|1 + \Delta t\,\lambda_k| \le 1 \quad \text{for all } k,
$$

which leads to a strict time-step restriction.



**Backward Euler:**

$$
g_k = \frac{1}{1 - \Delta t\,\lambda_k}
$$

Since $\lambda_k < 0$, the denominator is always larger than 1, so:

$$
|g_k| < 1 \quad \text{for all } \Delta t > 0.
$$

Backward Euler is therefore **unconditionally stable**.



**Crank–Nicolson:**

$$
g_k = \frac{1 + \tfrac12 \Delta t\,\lambda_k}{1 - \tfrac12 \Delta t\,\lambda_k}
$$

For diffusion operators, this also satisfies $|g_k| \le 1$ for all $\Delta t$.



This explains why:

👉 **Implicit methods remain stable even for very large time steps**, while explicit methods do not.


---

## 14. Note on numerical damping

Backward Euler remains stable for arbitrarily large time steps, but introduces **excess numerical damping of high-frequency spatial modes**.

As a result:

- fine spatial oscillations disappear faster than in the exact solution,
- the solution may become overly smooth for large $\Delta t$.

Crank–Nicolson preserves these modes more accurately but is slightly more expensive computationally.


---
## 15. Implementations of Eigenvalues, eigenmodes, and spectral interpretation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Parameters and grid

L = 1.0          # domain length
N = 40           # number of interior grid points
alpha = 1.0      # diffusion coefficient

x = np.linspace(0, L, N + 2)           # spatial discretization including boundaries x=0 and x=1
x_interior = x[1:-1]                   # spatial discretization excluding boundaries x=0 and x=1
h =  L / (N + 1)                       # spatial step


In [ ]:
# Build discrete 1D Laplacian matrix A (Dirichlet Boundary Conditions)

A = (1.0 / h**2) * (
    np.diag(-2.0 * np.ones(N))
    + np.diag(1.0 * np.ones(N - 1), k=1)
    + np.diag(1.0 * np.ones(N - 1), k=-1)
    )

print(A)

In [ ]:
# Eigenvalue decomposition of A

# Eigenvalue equation: A*v_k = lambda_k*v_k                   
lambda_vals, V = np.linalg.eigh(A)                 # eigenvalues, eigenvectors

# Ensure ascending order (eigh usually does this, but we enforce it)
idx = np.argsort(lambda_vals)[::-1]

# Heat equation operator eigenvalues: lambda = alpha * lambda_vals
lambda_vals = alpha * lambda_vals


In [ ]:
# Print first few eigenvalues

for k in range(5):
    print(f"Mode {k+1}: lambda = {lambda_vals[k]:.3f}")

# Verify orthonormality numerically: V^T*V ≈ I
orthogonality_check = V.T @ V
print("Max deviation from identity:", np.max(np.abs(orthogonality_check - np.eye(N))))

In [ ]:
# Eigenvalues from largest to smallest

lambda_vals = lambda_vals[idx]            
V = V[:, idx]

# Fix eigenvector sign ambiguity 
def fix_sign(eigvecs):
    for i in range(eigvecs.shape[1]):
        if eigvecs[1, i] < 0:
            eigvecs[:, i] *= -1
    return eigvecs

V = fix_sign(V)

# Analytical eigenfunctions for comparison (continuous)
def analytical_dirichlet(x, k, L=1.0):
    # k = 1,2,3,...
    return np.sin(k * np.pi * x / L)

In [ ]:
# Plotting thhree first Dirichlet eigenmodes

modes_to_plot = 3

x_D = np.linspace(0, 1, N+2)

plt.figure(figsize=(12, 8))

for k in range(modes_to_plot):

    # ------- DIRICHLET --------- #
    
    mode_D = np.zeros(N+2)
    mode_D[1:-1] = V[:, k]

    ana_D = analytical_dirichlet(x_D, k+1)

    # normalize analytical to numerical amplitude
    ana_D = ana_D / np.max(np.abs(ana_D)) * np.max(np.abs(mode_D))

    plt.subplot(2, modes_to_plot, k+1)
    plt.plot(x_D, mode_D, ".", label="Numerical")
    plt.plot(x_D, ana_D, linewidth=1, label="Analytical")
    plt.title(f"Dirichlet mode {k+1}\nλ={lambda_vals[k]:.2f}")
    plt.grid(True)

    if k == 0:
        plt.legend()

 
plt.tight_layout()
plt.show()


---

**Spectral solution of the heat equation:**
$
\,\, \mathbf{u}(t) = \sum_k c_k(0)\, e^{\alpha \lambda_k t}\,\mathbf{v}_k
$


In [ ]:
# Spectral solution of the heat equation
# u(x,t) = sum_k[c_k(0)*exp(lambda_k t)*v_k(x)]

# Example initial condition (u0)
u0 = np.sin(np.pi * x_interior / L) + 0.5 * np.sin(3 * np.pi * x_interior / L)

# Project initial condition onto eigenmodes
c = V.T @ u0


# Time evolution using spectral representation
# Using a discrete timestep dt

dt = 0.01
T_final = 0.2
times = np.arange(0.0, T_final + dt, dt)

def solution_spectral_time_series(times):
    U = []
    for t in times:
        U.append(V @ (c * np.exp(lambda_vals * t)))
    return np.array(U)

U_series = solution_spectral_time_series(times)

# Plot solution at selected timesteps
plot_indices = [0, int(0.01/dt), int(0.05/dt), int(0.2/dt)]

plt.figure(figsize=(7, 5))
for idx in plot_indices:
    plt.plot(x_interior, U_series[idx], label=f"t = {times[idx]:.3f}")

plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.title("Heat equation solution via eigenmode expansion (discrete dt)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

**Explanation:** Each coefficient evolves independently. Low-frequency modes decay slowly. High-frequency (oscillatory) modes decay rapidly. As a result, the temperature distribution evens out over time.

In [ ]:
# Visualize decay of modal amplitudes (discrete time)

plt.figure(figsize=(7, 5))
for k in range(4):
    plt.plot(times, c[k] * np.exp(lambda_vals[k] * times), "o-", label=f"mode {k+1}")

plt.xlabel("time")
plt.ylabel("modal amplitude")
plt.title("Exponential decay of eigenmodes (sampled with dt)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

#### Key physical message: ####

✅  **negative eigenvalues → exponential decay of modes.**

Heikki Miettinen 2026